In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [3]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [4]:
# 0: HAM
# 1: SPAM

from sklearn.model_selection import train_test_split

X = data.text
y = data.label
seed = 13

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=seed)

In [5]:
X_test

945    Mr ABDUL  MUSAThe Manager of Audit & Accountan...
452    Welcome back. I changed my plans and will be b...
304    Attached and enclosed below is the demarche an...
433    How does the article compare to the cover???Hu...
864    8:15 am DEPART Private Residence *En route Sta...
                             ...                        
999    sbwhoeopSunday February 21 2010 7:42 PMHShaunH...
191    Good morning1. Cheryl is trying to reach you t...
906    Ã¢ÂÂ Declassify on: 04/02/2025Readout of con...
508    H <hrod17@clintonemail.com>Tuesday December 22...
865    10; 127, 1999]" src=3D"http://www.crwflags.com...
Name: text, Length: 250, dtype: str

## Data Preprocessing

In [6]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [7]:
import re

def clean_html_for_nlp(text):
    # Step 1: Remove <script>...</script> and <style>...</style> blocks
    text = re.sub(r'<(script|style)[^>]*>.*?</(script|style)>', ' ', text,
                  flags=re.IGNORECASE | re.DOTALL)

    # Step 2: Remove HTML comments <!-- ... --> BEFORE stripping tags
    text = re.sub(r'<!--.*?-->', ' ', text, flags=re.DOTALL)

    # Step 3: Remove all remaining HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    return text



In [8]:
def decode_html_entities(text):
    # Decode common HTML entities
    html_entities = {
        '&nbsp;': ' ', '&amp;': '&', '&lt;': '<',
        '&gt;': '>', '&quot;': '"', '&#39;': "'"
    }
    for entity, char in html_entities.items():
        text = text.replace(entity, char)

    # Collapse all whitespace to single spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [9]:
import quopri

def decode_quopri(text: str) -> str:
    # quopri opera em bytes → encode primeiro
    raw_bytes = text.encode('utf-8', errors='replace')
    decoded_bytes = quopri.decodestring(raw_bytes)

    # Tenta codificações em ordem de probabilidade para emails antigos
    for encoding in ('utf-8', 'latin-1', 'cp1252'):
        try:
            return decoded_bytes.decode(encoding)
        except UnicodeDecodeError:
            continue

    # Último recurso: substitui caracteres problemáticos por '?'
    return decoded_bytes.decode('utf-8', errors='replace')


In [10]:
def remove_markdown_and_urls(text: str) -> str:
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', ' ', text)  # [texto](url)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)         # URLs soltas
    text = re.sub(r'\S+@\S+', ' ', text)                  # emails soltos
    return text

In [11]:
def clean_data(text):
    text = decode_quopri(text)
    text = clean_html_for_nlp(text)
    text = decode_html_entities(text)
    text = remove_markdown_and_urls(text)
    return text

In [12]:
X_train = X_train.apply(clean_data)
X_test = X_test.apply(clean_data)

In [13]:
X_test

945    Mr ABDUL MUSAThe Manager of Audit & Accountanc...
452    Welcome back. I changed my plans and will be b...
304    Attached and enclosed below is the demarche an...
433    How does the article compare to the cover???Hu...
864    8:15 am DEPART Private Residence *En route Sta...
                             ...                        
999    sbwhoeopSunday February 21 2010 7:42 PMHShaunH...
191    Good morning1. Cheryl is trying to reach you t...
906    Ã¢ÂÂ Declassify on: 04/02/2025Readout of con...
508    H Tuesday December 22 2009 10:04 AMB6And it wa...
865    10; 127, 1999]" src="  bord=er=0> &=nbsp; Dear...
Name: text, Length: 250, dtype: str

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [14]:

def clean_text(text):
    
    # remove prefix b''
    # b at the beginning and ' in the end
    text = re.sub(r"^b'|'$", " ", text)

    # remove special characters and numbers
    text  = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # remove single characters
    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)

    # Replace multiple spaces by single space
    text = re.sub(r'\s+', ' ', text).strip()

    # convert to lower case
    text = text.lower()

    return text


In [15]:
X_train = X_train.apply(clean_text)
X_test = X_test.apply(clean_text)

In [16]:
X_train[:10]

115    am traveling outside of the country with inter...
168    of kin and have them release the deposit to yo...
56     tom harkin called office twice duringcell home am
117                        book will be there in minutes
671    ebeling betsywednesday july pmre said yeah yea...
0      dear sir strictly private business proposal am...
37     dear ceo am mr joseph abu rime regional superv...
766    feel quite safe dealing with you in this impor...
709    think mentioned to you that carols pascual has...
157                                 services in post doc
Name: text, dtype: str

In [17]:
X_test[:10]

945    mr abdul musathe manager of audit accountancy ...
452    welcome back changed my plans and will be back...
304    attached and enclosed below is the demarche an...
433    how does the article compare to the cover huma...
864    am depart private residence en route state dep...
291    thx for sending the emails about the nigerian ...
986    he was larger than life in everything he did f...
923       we have to come up with something which we can
260                           fyi traffic from bottom up
546                    and left this return phone number
Name: text, dtype: str

## Now let's work on removing stopwords
Remove the stopwords.

In [18]:
def remove_stopwords(text):
    stop_words = stopwords.words("english")

    final_text = []

    for word in text.split(" "):

        if word not in stop_words:
            final_text.append(word)

    return " ".join(final_text)
    

In [19]:
X_train = X_train.apply(remove_stopwords)
X_test = X_test.apply(remove_stopwords)

In [20]:
X_train[:10]

115    traveling outside country intermittent access ...
168    kin release deposit weshare proceeds would gon...
56        tom harkin called office twice duringcell home
117                                         book minutes
671    ebeling betsywednesday july pmre said yeah yea...
0      dear sir strictly private business proposal mi...
37     dear ceo mr joseph abu rime regional superviso...
766    feel quite safe dealing important business tho...
709    think mentioned carols pascual asked work publ...
157                                    services post doc
Name: text, dtype: str

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [21]:
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

def get_wordnet_pos(word):
    """Map POS tag to first character lemmatize() accepts"""
    tag = nltk.pos_tag([word])[0][1][0]#  Get POS tag's first character (e.g., 'N' from 'NN')
    #Maps it to a WordNet-compatible tag
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    
    return tag_dict.get(tag, wordnet.NOUN) # returns the word type (Noun if we have not found)


def lemmatize(text):

    final_text = []
    wordnet_lemma = WordNetLemmatizer()

    for word in text.split(" "):
        lemmatized = wordnet_lemma.lemmatize(word, get_wordnet_pos(word))
        final_text.append(lemmatized)

    return final_text



In [22]:
sample = X_train[0]

print(sample)

out = lemmatize(sample)

print(" ".join(out))


dear sir strictly private business proposal mike chukwu manager bills exchange foreign remittance department zenith international bank plc writing letter ask support cooperation carry business opportunity department discovered abandoned sum fifteen million united states dollars account belongs one foreign customers died along entire family wife two children november plane crash since heard death expecting next kin come put claims money heir cannot release fund account unless someone applies claim next kin deceased indicated banking guidelines unfortunately neither family member distant relative ever appeared claim said fund upon discovery officials department agreed make business release total amount account heir fund since one came discovered maintained account bank otherwise fund returned banks treasury unclaimed fund agreed ratio sharing stated thus foreign partner us officials department settlement local foreign expences incurred us course business upon successful completion transf

In [23]:
X_train = X_train.apply(lemmatize)
X_test = X_test.apply(lemmatize)

In [24]:
# 3. O que entra no vectorizer após o apply
print("=== X_train após apply ===")
print(f"type(X_train[0]): {type(X_train.iloc[0])}")
print(f"valor:            {repr(X_train.iloc[0][:300])}")
print()


=== X_train após apply ===
type(X_train[0]): <class 'list'>
valor:            ['travel', 'outside', 'country', 'intermittent', 'access', 'email', 'need', 'reach', 'urgently', 'pleasecontact', 'nora', 'toiv', 'joanne', 'laszczych', 'thank', 'youcdm']



## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [25]:
from sklearn.feature_extraction.text import CountVectorizer

    # Convert processed_docs to strings
docs_as_strings = [' '.join(doc) for doc in X_train]

# Create the Bag of Words model
vectorizer = CountVectorizer(max_features=5000)
X_out = vectorizer.fit_transform(docs_as_strings)

# Show the Bag of Words feature names and the document-term matrix
feature_names = vectorizer.get_feature_names_out()
word_count = X_out.toarray()



In [26]:
feature_names

array(['aa', 'aal', 'ab', ..., 'zy', 'zywn', 'zz'],
      shape=(5000,), dtype=object)

In [27]:
len(docs_as_strings)

750

In [28]:
word_count # shape: n_documents x n_words

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(750, 5000))

In [29]:
import pandas as pd
count_df = pd.DataFrame(word_count, columns = feature_names)
count_df

,aa,aal,ab,abacha,abandon,abbas,abdul,abdull,abedin,abidjan,...,zsb,zt,zv,zw,zwx,zx,zxj,zy,zywn,zz
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
746,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
747,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
748,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [30]:
temp_words = [' '.join(doc) for doc in X_test]
word_count_test = vectorizer.transform(temp_words)
count_df_test =  pd.DataFrame(word_count_test.toarray(), columns = feature_names)

In [31]:
count_df_test

,aa,aal,ab,abacha,abandon,abbas,abdul,abdull,abedin,abidjan,...,zsb,zt,zv,zw,zwx,zx,zxj,zy,zywn,zz
0,0,0,0,0,2,0,2,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
246,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
247,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
248,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
# 0: HAM
# 1: SPAM

ham_label = 0
spam_label = 1

ham_index =  y_train.reset_index(drop=True) == ham_label
spam_index = y_train.reset_index(drop=True) == spam_label


In [33]:
ham_index

0       True
1      False
2       True
3       True
4       True
       ...  
745    False
746     True
747    False
748    False
749     True
Name: label, Length: 750, dtype: bool

In [34]:
result = (count_df[ham_index]
          .sum()
          .to_frame(name="ham_count")
          .rename_axis("word")
          .reset_index()
          .sort_values(by="ham_count", ascending=False)[:10])

result

,word,ham_count
635,call,90
3911,say,86
4111,state,85
3435,president,82
3255,percent,80
2982,obama,78
4880,would,74
3341,pm,72
4873,work,63
1758,get,62


In [35]:
result = (count_df[spam_index]
          .sum()
          .to_frame(name="spam_count")
          .rename_axis("word")
          .reset_index()
          .sort_values(by="spam_count", ascending=False)[:10])

result

,word,spam_count
2740,money,756
48,account,712
475,bank,609
1683,fund,594
4417,transfer,449
4415,transaction,403
621,business,398
896,country,372
2780,mr,367
2687,million,365


## Extra features

In [36]:
X_train

115    [travel, outside, country, intermittent, acces...
168    [kin, release, deposit, weshare, proceeds, wou...
56     [tom, harkin, call, office, twice, duringcell,...
117                                       [book, minute]
671    [ebeling, betsywednesday, july, pmre, say, yea...
                             ...                        
742    [dear, friend, mr, mark, boland, bank, manager...
528    [draft, introductory, letter, qddr, jake, anne...
74     [good, morning, offend, may, disturbed, daily,...
176    [dearest, found, seriousness, iswhy, decide, i...
338                                               [know]
Name: text, Length: 750, dtype: object

In [37]:
X_train_bow = X_train.reset_index(drop=True)
X_test_bow = X_test.reset_index(drop=True)

In [38]:
X_train_bow = X_train_bow.to_frame(name="preprocessed_text")
X_test_bow = X_test_bow.to_frame(name="preprocessed_text")

In [39]:
X_train_bow

,preprocessed_text
0,"[travel, outside, country, intermittent, acces..."
1,"[kin, release, deposit, weshare, proceeds, wou..."
2,"[tom, harkin, call, office, twice, duringcell,..."
3,"[book, minute]"
4,"[ebeling, betsywednesday, july, pmre, say, yea..."
...,...
745,"[dear, friend, mr, mark, boland, bank, manager..."
746,"[draft, introductory, letter, qddr, jake, anne..."
747,"[good, morning, offend, may, disturbed, daily,..."
748,"[dearest, found, seriousness, iswhy, decide, i..."


In [40]:
money_simbol_list = ["euro","dollar","pound"]
suspicious_words = ["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"]

print(money_simbol_list)
print(suspicious_words)

['euro', 'dollar', 'pound']
['free', 'cheap', 'sex', 'money', 'account', 'bank', 'fund', 'transfer', 'transaction', 'win', 'deposit', 'password']


In [41]:
# Train set
X_train_bow['money_mark'] = (X_train_bow['preprocessed_text']
                               .apply(
                                   lambda words_list : len(set(words_list) & set(money_simbol_list))
                               ))


X_train_bow['suspicious_words'] = (X_train_bow['preprocessed_text']
                                     .apply(
                                         lambda word_list : len(set(word_list) & set(suspicious_words))
                                     ))

X_train_bow['n_words'] = (X_train_bow['preprocessed_text']
                             .apply(
                                 lambda word_list : len(word_list)
                             )
                             )

# Test set
X_test_bow['money_mark'] = (X_test_bow['preprocessed_text']
                               .apply(
                                   lambda words_list : len(set(words_list) & set(money_simbol_list))
                               ))


X_test_bow['suspicious_words'] = (X_test_bow['preprocessed_text']
                                     .apply(
                                         lambda word_list : len(set(word_list) & set(suspicious_words))
                                     ))

X_test_bow['n_words'] = (X_test_bow['preprocessed_text']
                             .apply(
                                 lambda word_list : len(word_list)
                             )
                             )



In [42]:
X_train_bow

,preprocessed_text,money_mark,suspicious_words,n_words
0,"[travel, outside, country, intermittent, acces...",0,0,16
1,"[kin, release, deposit, weshare, proceeds, wou...",0,3,38
2,"[tom, harkin, call, office, twice, duringcell,...",0,0,7
3,"[book, minute]",0,0,2
4,"[ebeling, betsywednesday, july, pmre, say, yea...",0,0,8
...,...,...,...,...
745,"[dear, friend, mr, mark, boland, bank, manager...",2,6,284
746,"[draft, introductory, letter, qddr, jake, anne...",0,0,12
747,"[good, morning, offend, may, disturbed, daily,...",1,7,249
748,"[dearest, found, seriousness, iswhy, decide, i...",1,5,120


In [43]:
extra_features_train = X_train_bow.drop("preprocessed_text", axis=1)
extra_features_test = X_test_bow.drop("preprocessed_text", axis=1)

In [44]:
X_train_bow = pd.concat([extra_features_train, count_df], axis=1)
X_train_bow

,money_mark,suspicious_words,n_words,aa,aal,ab,abacha,abandon,abbas,abdul,...,zsb,zt,zv,zw,zwx,zx,zxj,zy,zywn,zz
0,0,0,16,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,3,38,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,7,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,8,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745,2,6,284,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
746,0,0,12,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
747,1,7,249,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
748,1,5,120,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [45]:
X_test_bow = pd.concat([extra_features_test, count_df_test], axis=1)
X_test_bow

,money_mark,suspicious_words,n_words,aa,aal,ab,abacha,abandon,abbas,abdul,...,zsb,zt,zv,zw,zwx,zx,zxj,zy,zywn,zz
0,0,6,180,0,0,0,0,2,0,2,...,0,0,0,0,0,0,0,0,0,0
1,0,0,30,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,2,470,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,128,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,0,0,18,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
246,0,0,36,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
247,0,0,12,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
248,0,0,28,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [46]:
X_train_bow.shape

(750, 5003)

In [47]:
X_test_bow.shape

(250, 5003)

In [48]:
# # We add to the original dataframe two additional indicators (money symbols and suspicious words).
# money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
# suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

# data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
# data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
# data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

# data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
# data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
# data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

# data_train.head()

## How would work the Bag of Words with Count Vectorizer concept?

In [49]:
# The previous bag-of-words were conducted using CountVectorizer. First we instantiate the vectorizer, fit it and then apply to transform the data.

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [50]:
temp_words = [' '.join(doc) for doc in X_train]
vectorizer = TfidfVectorizer(max_features=5000)
word_count = vectorizer.fit_transform(temp_words)

count_df =  pd.DataFrame(word_count.toarray(), columns = feature_names)
count_df

,aa,aal,ab,abacha,abandon,abbas,abdul,abdull,abedin,abidjan,...,zsb,zt,zv,zw,zwx,zx,zxj,zy,zywn,zz
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
746,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
747,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
748,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [52]:
X_train_tfidf = pd.concat([extra_features_train, count_df], axis=1)
X_train_tfidf

,money_mark,suspicious_words,n_words,aa,aal,ab,abacha,abandon,abbas,abdul,...,zsb,zt,zv,zw,zwx,zx,zxj,zy,zywn,zz
0,0,0,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,3,38,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,0,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,0,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,0,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745,2,6,284,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
746,0,0,12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
747,1,7,249,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
748,1,5,120,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
temp_words = [' '.join(doc) for doc in X_test]
word_count_test = vectorizer.transform(temp_words)

count_df_test =  pd.DataFrame(word_count_test.toarray(), columns = feature_names)
count_df_test

,aa,aal,ab,abacha,abandon,abbas,abdul,abdull,abedin,abidjan,...,zsb,zt,zv,zw,zwx,zx,zxj,zy,zywn,zz
0,0.0,0.0,0.0,0.0,0.150789,0.0,0.214898,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
246,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
247,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
248,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [53]:
X_test_tfidf = pd.concat([extra_features_test, count_df_test], axis=1)
X_test_tfidf

,money_mark,suspicious_words,n_words,aa,aal,ab,abacha,abandon,abbas,abdul,...,zsb,zt,zv,zw,zwx,zx,zxj,zy,zywn,zz
0,0,6,180,0,0,0,0,2,0,2,...,0,0,0,0,0,0,0,0,0,0
1,0,0,30,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,2,470,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,128,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,0,0,18,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
246,0,0,36,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
247,0,0,12,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
248,0,0,28,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## And the Train a Classifier?

### bow


In [54]:
from xgboost import XGBClassifier

xgb = XGBClassifier(eval_metric="mlogloss")

xgb.fit(X_train_bow, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [55]:
from sklearn.metrics import accuracy_score

y_pred_train = xgb.predict(X_train_bow)

accuracy_score(y_train, y_pred_train)

0.9853333333333333

In [56]:
y_pred_test = xgb.predict(X_test_bow)
accuracy_score(y_test, y_pred_test)

0.952

### TF-IDF

In [58]:
xgb = XGBClassifier(eval_metric="mlogloss")

xgb.fit(X_test_tfidf, y_test)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [59]:
y_pred_train = xgb.predict(X_train_tfidf)

accuracy_score(y_train, y_pred_train)

0.9133333333333333

In [60]:
y_pred_test = xgb.predict(X_test_tfidf)

accuracy_score(y_test, y_pred_test)

0.984

### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code